In [2]:
import os
import json
import re
from pathlib import Path
from collections import OrderedDict
import xml.etree.ElementTree as ET

# ── 1. CONFIGURATION PATHS & PAGE WINDOWS ───────────────────────────────────
HATHI_OCR_PATH = "/Users/gcrane/Downloads/aristotle-poetics-twining-uc2-ark--13960-t2s46z04n-1779495726.txt" 
TEI_PATH = "/Users/gcrane/github/Poetics2.0/eng/tlg0086.tlg034.twining1789-eng1.xml"

# Specify the integer page numbers you want to include (Inclusive)
# Set to None if you want to disable boundaries and scan the whole file
START_PAGE = 65
STOP_PAGE = 131

WORKSPACE_DIR = Path("./dist") 
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

NS = {'tei': 'http://www.tei-c.org/ns/1.0'}

WORK_REGISTRY = {
    "tlg0003.tlg001": {
        "textgroup": "tlg0003",
        "work": "tlg001",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.perseus-grc2.xml", "label": "Greek (H. S. Jones, 1942)", "class": "greek-text"},
            "1st1K-eng1": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-eng1.xml", "label": "English (C. F. Smith, 1919)", "class": "english-text"},
            "perseus-eng6": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.perseus-eng6.xml", "label": "English (R. Crawley, 1914)", "class": "english-text"},
            "1st1k-fre1": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1k-fre1.xml", "label": "French (E. Bétant, 1863)", "class": "french-text"},
            "1st1k-lat2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1k-lat2.xml", "label": "Latin (Fr. Haase, 1869)", "class": "latin-text"}
        }
    },
    "tlg0086.tlg034": {
        "textgroup": "tlg0086",
        "work": "tlg034",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.perseus-grc2.xml", "label": "Greek (Kassel, 1965)", "class": "greek-text"},
            "digicorpus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.digicorpus-grc2.xml", "label": "Greek (Digital Corpus Variant)", "class": "greek-text"},
            "perseus-eng2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.perseus-eng2.xml", "label": "English (W.H. Fyfe, 1927)", "class": "english-text"},
            "butcher1911-eng2": {"path": "/Users/gcrane/github/Poetics2.0/grc/tlg0086.tlg034.butcher1911-eng2.xml", "label": "English (S.H. Butcher, 1911)", "class": "english-text"},
            "bywater1909-eng1": {"path": "/Users/gcrane/github/Poetics2.0/grc/tlg0086.tlg034.bywater1909-eng1.xml", "label": "English (Ingram Bywater, 1909)", "class": "english-text"},
            "twining1789-eng1": {"path": TEI_PATH, "label": "English (Thomas Twining, 1789)", "class": "english-text", "parse_mode": "milestones"}
        }
    },
    "tlg0011.tlg004": {
        "textgroup": "tlg0011",
        "work": "tlg004",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg004/tlg0011.tlg004.perseus-grc2.xml", "label": "Greek (F. Storr, 1912)", "class": "greek-text", "parse_mode": "poetry_cards"},
            "perseus-eng2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg004/tlg0011.tlg004.perseus-eng2.xml", "label": "English (Sir Richard Jebb, 1904)", "class": "english-text", "parse_mode": "poetry_cards"}
        }
    }
}

def generate_canonical_id(work_key, label_string):
    prefix = work_key.replace('.', '_')
    match = re.search(r'\((.*?)\)', label_string)
    if match:
        content = match.group(1)
        parts = content.split(',')
        editor = parts[0].strip()
        year = parts[1].strip() if len(parts) > 1 else ""
        
        editor_last = editor.split()[-1].lower().replace('.', '')
        year_clean = re.sub(r'\D', '', year)
        return f"{prefix}_{editor_last}_{year_clean}"
    
    fallback_suffix = re.sub(r'\W+', '_', label_string).lower()
    return f"{prefix}_{fallback_suffix}"

def find_text_root(root):
    for div in root.findall('.//{http://www.tei-c.org/ns/1.0}div') + root.findall('.//div'):
        if div.get('type') == 'translation': return div
    return root.find('.//{http://www.tei-c.org/ns/1.0}body') or root.find('.//body') or root.find('.//*body')

def extract_text_recursive(elem, strip_paragraphs=False):
    parts = []
    tag = elem.tag.split('}')[-1]
    
    if tag == 'l':
        line_num = (elem.get('n') or '').strip()
        parts.append(f'<div class="verse-line" data-line="{line_num}">')
    elif tag == 'speaker': 
        parts.append('<strong class="speaker-attr">')
    elif tag == 'stage':
        parts.append('<div class="stage-direction">')
    elif tag == 'hi':
        rend = elem.get('rend', 'italic')
        parts.append(f'<span class="render-{rend}">')
    elif tag == 'quote':
        q_type = elem.get('type', 'blockquote')
        parts.append(f'<blockquote class="quote-block type-{q_type}">')
    elif tag == 'p' and not strip_paragraphs:
        parts.append('<p>')

    if elem.text: parts.append(elem.text)
    for child in elem:
        child_tag = child.tag.split('}')[-1]
        if child_tag == 'note':
            note_text = extract_text_recursive(child, strip_paragraphs).strip()
            if note_text: parts.append(f'<span class="note">[{note_text}]</span>')
        elif child_tag == 'lb': parts.append('<br/>')
        else: parts.append(extract_text_recursive(child, strip_paragraphs))
        if child.tail: parts.append(child.tail)
            
    if tag == 'l': parts.append('</div>')
    elif tag == 'speaker': parts.append(': </strong>')
    elif tag == 'stage': parts.append('</div>')
    elif tag == 'hi': parts.append('</span>')
    elif tag == 'quote': parts.append('</blockquote>')
    elif tag == 'p' and not strip_paragraphs: parts.append('</p>')
    return ''.join(parts)

def parse_hierarchical_tei(path):
    if not os.path.exists(path): return None
    tree = ET.parse(path)
    root = tree.getroot()
    text_entry = find_text_root(root)
    if text_entry is None: return None
    data = OrderedDict()

    def walk_divisions(node, current_path):
        tag = node.tag.split('}')[-1]
        subtype = node.get('subtype') or node.get('type')
        n_val = node.get('n')

        if tag == 'div' and subtype in ('book', 'chapter', 'section', 'part', 'textpart') and n_val:
            resolved_subtype = 'chapter' if subtype in ('part', 'textpart') and n_val.isdigit() else subtype
            new_path = current_path + [(resolved_subtype, str(n_val).strip())]
        else:
            new_path = current_path

        paragraphs = node.findall('{http://www.tei-c.org/ns/1.0}p') or node.findall('p')
        if paragraphs and new_path:
            key_map = {t: v for t, v in new_path}
            bk = key_map.get('book', '1')
            ch = key_map.get('chapter', key_map.get('part', '1'))
            sec = key_map.get('section', n_val or '1')

            if bk not in data: data[bk] = OrderedDict()
            if ch not in data[bk]: data[bk][ch] = OrderedDict()
            
            combined_txt = ' '.join(extract_text_recursive(p, strip_paragraphs=False).strip() for p in paragraphs)
            if combined_txt: data[bk][ch][sec] = combined_txt
            return

        for child in node: walk_divisions(child, new_path)

    walk_divisions(text_entry, [])
    return data

def parse_milestone_aligned_tei(path):
    if not os.path.exists(path): return None
    tree = ET.parse(path)
    root = tree.getroot()
    text_entry = find_text_root(root)
    if text_entry is None: return None

    data = OrderedDict()
    current_ch = "1"
    current_sec = "1"
    current_text = []

    def _flush():
        nonlocal current_ch, current_sec, current_text
        if current_text:
            bk = '1'
            data.setdefault(bk, OrderedDict()).setdefault(current_ch, OrderedDict())
            html_blob = ' '.join(t.strip() for t in current_text if t.strip())
            if html_blob:
                if current_sec in data[bk][current_ch]:
                    data[bk][current_ch][current_sec] += " " + html_blob
                else:
                    data[bk][current_ch][current_sec] = html_blob
        current_text = []

    def _walk(elem):
        nonlocal current_ch, current_sec, current_text
        tag = elem.tag.split('}')[-1]

        if tag == 'milestone':
            unit = elem.get('unit') or ''
            val = (elem.get('n') or '').strip()
            
            if val:
                if unit == 'bekker' and '.' in val:
                    parts = val.split('.', 1)
                    _flush()
                    current_ch = parts[0]
                    current_sec = parts[1]
                else:
                    num_match = re.match(r'^\d+', val)
                    if num_match:
                        if unit in ('chapter', 'chapter-start', ''):
                            _flush()
                            current_ch = str(int(num_match.group(0)))
                            current_sec = "1"
                        elif unit in ('section', 'section-start'):
                            _flush()
                            current_sec = str(int(num_match.group(0)))
                        
            if elem.tail and elem.tail.strip(): current_text.append(elem.tail)
            return

        if tag == 'note':
            note_html = extract_text_recursive(elem, strip_paragraphs=True).strip()
            if note_html: current_text.append(f'<span class="note">[{note_html}]</span>')
            if elem.tail and elem.tail.strip(): current_text.append(elem.tail)
            return

        if tag in ('hi', 'foreign', 'emph'):
            rend = elem.get('rend', 'italic')
            current_text.append(f'<span class="render-{rend}">')
            if elem.text: current_text.append(elem.text)
            for child in elem: _walk(child)
            current_text.append('</span>')
            if elem.tail and elem.tail.strip(): current_text.append(elem.tail)
            return

        if tag == 'p' or tag == 'div':
            if elem.text and elem.text.strip(): current_text.append(elem.text)
            for child in elem: _walk(child)
            if elem.tail and elem.tail.strip(): current_text.append(elem.tail)
            return

        if elem.text and elem.text.strip(): current_text.append(elem.text)
        for child in elem: _walk(child)
        if elem.tail and elem.tail.strip(): current_text.append(elem.tail)

    _walk(text_entry)
    _flush()  
    return data

def build_poetry_canonical_intervals(editions_dict):
    grc_cfg = editions_dict.get('perseus-grc2') or list(editions_dict.values())[0]
    tree = ET.parse(grc_cfg["path"])
    text_entry = find_text_root(tree.getroot())
    
    landmarks = []
    for elem in text_entry.iter():
        if elem.tag.endswith('milestone') and elem.get('unit') == 'card':
            landmarks.append(('card', elem.get('n').strip()))
        elif elem.tag.endswith('l'):
            ln = (elem.get('n') or '').strip()
            if ln: landmarks.append(('line', ln))

    intervals = []
    for idx, item in enumerate(landmarks):
        if item[0] == 'card':
            card_n = item[1]
            first_l, last_l = None, None
            for ahead in landmarks[idx+1:]:
                if ahead[0] == 'card': break
                if ahead[0] == 'line':
                    if first_l is None: first_l = ahead[1]
                    last_l = ahead[1]
            if not first_l: first_l = card_n
            if not last_l: last_l = first_l
            intervals.append({'card_n': card_n, 'label': f"{first_l}-{last_l}"})
    return intervals

def parse_poetry_cards_tei(path, master_intervals):
    if not os.path.exists(path): return None
    tree = ET.parse(path)
    text_entry = find_text_root(tree.getroot())

    data = OrderedDict()
    data["1"] = OrderedDict()
    
    current_label = master_intervals[0]['label'] if master_intervals else "1"
    current_text = []

    def _flush():
        nonlocal current_label, current_text
        if current_text and current_label:
            html = ' '.join(t.strip() for t in current_text if t.strip())
            if html:
                if current_label not in data["1"]:
                    data["1"][current_label] = OrderedDict()
                data["1"][current_label]["1"] = html
        current_text = []

    def _walk(elem):
        nonlocal current_label, current_text
        tag = elem.tag.split('}')[-1]

        if tag == 'milestone' and elem.get('unit') == 'card':
            val = (elem.get('n') or '').strip()
            match = [r for r in master_intervals if r['card_n'] == val]
            if match:
                _flush()
                current_label = match[0]['label']
        elif tag == 'l' or tag == 'stage':
            current_text.append(extract_text_recursive(elem, strip_paragraphs=True))
            return
        elif tag == 'note':
            note_html = extract_text_recursive(elem, strip_paragraphs=True).strip()
            if note_html: current_text.append(f'<span class="note">[{note_html}]</span>')
            if elem.tail and elem.tail.strip(): current_text.append(elem.tail)
            return

        if elem.text and elem.text.strip(): current_text.append(elem.text)
        for child in elem: _walk(child)
        if elem.tail and elem.tail.strip(): current_text.append(elem.tail)

    _walk(text_entry)
    _flush()
    return data

GLOBAL_STRUCTURES = {}
GLOBAL_REGISTRIES = {}

for work_key, work_meta in WORK_REGISTRY.items():
    tg = work_meta["textgroup"]
    wk = work_meta["work"]
    editions = work_meta["editions"]
    
    print(f"Ingesting textual data layers for canonical identifier context: {work_key}...")
    
    for v_id, cfg in editions.items():
        canonical_id = generate_canonical_id(work_key, cfg["label"])
        GLOBAL_REGISTRIES[canonical_id] = {
            "urn": f"urn:cts:greekLit:{work_key}.{v_id}",
            "label": cfg["label"],
            "class": cfg["class"],
            "textgroup": tg,
            "work": wk,
            "short_id": v_id
        }

    master_intervals = None
    is_poetry = any(cfg.get("parse_mode") == "poetry_cards" for cfg in editions.values())
    if is_poetry:
        master_intervals = build_poetry_canonical_intervals(editions)

    work_corpus = OrderedDict()
    for v_id, cfg in editions.items():
        p_mode = cfg.get("parse_mode")
        if p_mode == "poetry_cards":
            parsed = parse_poetry_cards_tei(cfg["path"], master_intervals)
        elif p_mode == "milestones":
            parsed = parse_milestone_aligned_tei(cfg["path"])
        else:
            parsed = parse_hierarchical_tei(cfg["path"])
            
        if parsed is not None and sum(len(secs) for chs in parsed.values() for secs in chs.values()) > 0:
            work_corpus[v_id] = parsed
            print(f"  ✓ {v_id}: {sum(len(secs) for chs in parsed.values() for secs in chs.values())} segments parsed")
        else:
            print(f"  ✗ {v_id}: Failed to parse completely.")

    if not work_corpus: continue

    first_version = list(work_corpus.keys())[0]
    baseline_corpus = work_corpus[first_version]

    has_multiple_books = len(baseline_corpus.keys()) > 1
    structure_map = OrderedDict()
    chapter_sequence = []

    if has_multiple_books:
        for b_k, ch_v in baseline_corpus.items():
            structure_map[b_k] = list(ch_v.keys())
            for c_k in ch_v.keys():
                chapter_sequence.append({'book': b_k, 'chapter': c_k})
        GLOBAL_STRUCTURES[work_key] = structure_map
    else:
        single_bk_key = list(baseline_corpus.keys())[0]
        structure_map["_flat_chapters"] = list(baseline_corpus[single_bk_key].keys())
        for c_k in baseline_corpus[single_bk_key].keys():
            chapter_sequence.append({'book': None, 'chapter': c_k})
        GLOBAL_STRUCTURES[work_key] = structure_map["_flat_chapters"]

    print(f" -> Packaging structural formatted JS chunk scripts...")
    for c_idx, coord in enumerate(chapter_sequence):
        bk_id = coord['book']
        ch_id = coord['chapter']
        
        lookup_bk = bk_id if bk_id else list(baseline_corpus.keys())[0]
        baseline_secs = list(baseline_corpus[lookup_bk][ch_id].keys())
        sections_payload = OrderedDict()
        
        for sec in baseline_secs:
            sections_payload[sec] = {}
            for v_id in editions:
                ch_data = work_corpus.get(v_id, {}).get(lookup_bk, {}).get(ch_id, {})
                sections_payload[sec][v_id] = ch_data.get(sec, "<i>[Text range missing in alignment layer]</i>")

        if bk_id:
            passage_urn = f"urn:cts:greekLit:{work_key}:{bk_id}.{ch_id}"
            prev_urn = f"urn:cts:greekLit:{work_key}:{chapter_sequence[c_idx-1]['book']}.{chapter_sequence[c_idx-1]['chapter']}" if c_idx > 0 else None
            next_urn = f"urn:cts:greekLit:{work_key}:{chapter_sequence[c_idx+1]['book']}.{chapter_sequence[c_idx+1]['chapter']}" if c_idx < len(chapter_sequence) - 1 else None
            chunk_filename = f"chunk_b{bk_id}_ch{ch_id}.js"
        else:
            passage_urn = f"urn:cts:greekLit:{work_key}:{ch_id}"
            prev_urn = f"urn:cts:greekLit:{work_key}:{chapter_sequence[c_idx-1]['chapter']}" if c_idx > 0 else None
            next_urn = f"urn:cts:greekLit:{work_key}:{chapter_sequence[c_idx+1]['chapter']}" if c_idx < len(chapter_sequence) - 1 else None
            chunk_filename = f"chunk_ch{ch_id}.js"
        
        chapter_payload = {
            "urn": passage_urn,
            "textgroup": tg,
            "work": wk,
            "book": bk_id,
            "chapter": ch_id,
            "sections": sections_payload,
            "navigation": { "prev": prev_urn, "next": next_urn }
        }

        chunk_dir = WORKSPACE_DIR / "corpus" / tg / wk / "chunks"
        chunk_dir.mkdir(parents=True, exist_ok=True)
        
        js_wrapped = f"""/** Perseus Autonomous Text Chunk Module **/
registerWorkspaceChunk("{passage_urn}", {json.dumps(chapter_payload, indent=2, ensure_ascii=False)});
"""
        (chunk_dir / chunk_filename).write_text(js_wrapped, encoding='utf-8')

print("[SUCCESS] Data Ingestion Engine completed cleanly for all works.")

Ingesting textual data layers for canonical identifier context: tlg0003.tlg001...
  ✓ perseus-grc2: 3587 segments parsed
  ✓ 1st1K-eng1: 3580 segments parsed
  ✓ perseus-eng6: 3587 segments parsed
  ✓ 1st1k-fre1: 917 segments parsed
  ✓ 1st1k-lat2: 3624 segments parsed
 -> Packaging structural formatted JS chunk scripts...
Ingesting textual data layers for canonical identifier context: tlg0086.tlg034...
  ✓ perseus-grc2: 381 segments parsed
  ✓ digicorpus-grc2: 381 segments parsed
  ✓ perseus-eng2: 381 segments parsed
  ✓ butcher1911-eng2: 381 segments parsed
  ✓ bywater1909-eng1: 380 segments parsed
  ✓ twining1789-eng1: 380 segments parsed
 -> Packaging structural formatted JS chunk scripts...
Ingesting textual data layers for canonical identifier context: tlg0011.tlg004...
  ✓ perseus-grc2: 72 segments parsed
  ✓ perseus-eng2: 70 segments parsed
 -> Packaging structural formatted JS chunk scripts...
[SUCCESS] Data Ingestion Engine completed cleanly for all works.
